In [1]:
import pandas as pd
import numpy as np
import csv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(
    '../datos/dataset_nlp_utf16_final.csv', 
    encoding='utf-16', 
    sep=',', 
    quoting=csv.QUOTE_ALL, 
    escapechar='\\'
)

# Unificación de textos para comparar
# Juntamos lo que pide la empresa
df['texto_oferta'] = df['cargo'] + " " + df['funciones'] + " " + df['conocimientos'] + " " + df['experiencia']
# Juntamos lo que ofrece el candidato
df['texto_candidato'] = df['experiencia_cv'] + " " + df['estudios_cv']

# Trabajaremos los textos en minúsculas
df['texto_oferta'] = df['texto_oferta'].fillna('').str.lower()
df['texto_candidato'] = df['texto_candidato'].fillna('').str.lower()

print(f"Total de registros listos para análisis: {df.shape[0]}")

Total de registros listos para análisis: 220


In [2]:
# Trabajo adaptado para texto en español
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')  # Descarga del paquete de stop words
stop_words_es = stopwords.words('spanish')  # Crea una lista con las palabras vacías en español

print("Vectorizando textos con TF-IDF en español.")

vectorizador = TfidfVectorizer(max_features=5000, stop_words=stop_words_es)

# Ajusta el vocabulario usando tanto las ofertas como los CVs
corpus_completo = pd.concat([df['texto_oferta'], df['texto_candidato']])
vectorizador.fit(corpus_completo)

# Transforma textos a matrices numéricas
tfidf_ofertas = vectorizador.transform(df['texto_oferta'])
tfidf_candidatos = vectorizador.transform(df['texto_candidato'])

# Calcula la similitud del coseno fila por fila
similitudes = []
for i in range(df.shape[0]):
    # Calcula qué tan cerca está el vector del CV al vector de la Oferta (1 = Idéntico, 0 = Ninguna similitud)
    similitud = cosine_similarity(tfidf_ofertas[i], tfidf_candidatos[i])[0][0]
    similitudes.append(similitud)

df['score_similitud'] = similitudes
print("Similitud calculada. Muestra de los primeros 5 scores:")
print(df[['target_elegibilidad', 'score_similitud']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Vectorizando textos con TF-IDF en español.
Similitud calculada. Muestra de los primeros 5 scores:
   target_elegibilidad  score_similitud
0                    1         0.000000
1                    1         0.059231
2                    1         0.000000
3                    1         0.000000
4                    1         0.000000


In [5]:
# Diagnóstico de la Fila 0 (Similitud 0.000)
print("FILA 0 (Similitud 0.000)")
print("OFERTA:")
print(df['texto_oferta'].iloc[0])
print("\nCANDIDATO:")
print(df['texto_candidato'].iloc[0])

# Diagnóstico de la Fila 1 (Similitud 0.059)
print("OFERTA:")
print(df['texto_oferta'].iloc[1])
print("\nCANDIDATO:")
print(df['texto_candidato'].iloc[1])

FILA 0 (Similitud 0.000)
OFERTA:
analista programador junior formar parte de nuestros equipos de desarrollo, programando nuevas funcionalidades o adaptando funcionalidades existentes del erp a partir de requerimientos internos o de clientes. colaborar con la documentación de las funcionalidades existentes del erp, así como de las herramientas propietarias o de las diversas aplicaciones desarrolladas. participar en reuniones técnicas de programación y seguimiento. participar en los procesos de implementación y soporte de funcionalidades a clientes. conocimientos de java script, sql server (de preferencia). se valorará experiencia en desarrollo o uso de erps. se valorarán conocimientos de gestión de proyectos ágiles. con o sin experiencia

CANDIDATO:

OFERTA:
ingeniero de instrumentación desarrollar la ingeniería de su especialidad de instrumentación para los proyectos llevados a cabo por la cía, cumpliendo con los plazos y parámetros de calidad y costos establecidos. profesional titulad

In [6]:
# Calcula cantidades y porcentajes
total_filas = len(df)
filas_cero = (df['score_similitud'] == 0.0).sum()
porcentaje_ceros = (filas_cero / total_filas) * 100

print(f"Total de registros evaluados: {total_filas}")
print(f"Registros con similitud exacta de 0.0: {filas_cero}")
print(f"Porcentaje de ceros en el dataset: {porcentaje_ceros:.2f}%\n")

# Identifica cuáles son esos datos
df_ceros = df[df['score_similitud'] == 0.0]

if filas_cero > 0:
    columnas_clave = ['id_oferta', 'id_usuario', 'target_elegibilidad']
    print("Muestra de los primeros 10 registros con similitud 0.0:")
    display(df_ceros[columnas_clave].head(10))

Total de registros evaluados: 220
Registros con similitud exacta de 0.0: 190
Porcentaje de ceros en el dataset: 86.36%

Muestra de los primeros 10 registros con similitud 0.0:


,id_oferta,id_usuario,target_elegibilidad
0,34635,1002160,1
2,34733,10078,1
3,32671,6323,1
4,36568,1002976,1
5,31784,288,1
6,37342,11750,1
7,34274,1002949,1
9,36966,1004288,1
10,22499,6604,1
11,32973,11469,1
